In [0]:
WITH ranked_sellers AS (
    SELECT
        countryCode,
        identifierHash,
        gender,
        CAST(productsListed AS BIGINT) AS products_listed,
        CAST(productsSold AS BIGINT) AS products_sold,
        CAST(productsPassRate AS DOUBLE) AS pass_rate,
        CAST(socialNbFollowers AS BIGINT) AS followers,

        ROUND(
            100.0 * TRY_DIVIDE(
                CAST(productsSold AS DOUBLE),
                CAST(productsListed AS DOUBLE)
            ),
            2
        ) AS sell_through_pct,

        ROW_NUMBER() OVER (
            PARTITION BY countryCode
            ORDER BY
                CAST(productsSold AS BIGINT) DESC,
                CAST(productsPassRate AS DOUBLE) DESC,
                CAST(socialNbFollowers AS BIGINT) DESC
        ) AS seller_rank

    FROM ecommerce_fashion.gold.comprehensive_table
    WHERE LOWER(type) = 'user'
      AND (
          CAST(productsListed AS DOUBLE) > 0
          OR CAST(productsSold AS DOUBLE) > 0
      )
)
SELECT *
FROM ranked_sellers
WHERE seller_rank <= 10
ORDER BY countryCode, seller_rank;